# Sinhala POS Tagger & Robust Alignment Notebook

This notebook contains the complete solution for tagging Sinhala historical text and aligning it with the dataset.

**Key Improvements (v2):**
1.  **Beam Search TnT:** Fixes hanging on long sentences.
2.  **Sentence-Level Alignment:** Instead of a global stream, we aligning data **sentence-by-sentence** using `century` and `sentence_index` as keys. This ensures that even if one sentence has issues, the rest align perfectly.

## 1. Setup & Imports

In [1]:
import math
import csv
import json
import nltk
from collections import defaultdict
from nltk.tag import tnt
from nltk.probability import FreqDist, ConditionalFreqDist

## 2. Beam Search TnT Implementation

In [2]:
class BeamTnT(tnt.TnT):
    def __init__(self, unk=None, trained=False, N=1000):
        super().__init__(unk, trained)
        self.N = N # Beam width

    def tag(self, sent):
        # Iterative Beam Search decoding
        current_states = [ (["BOS", "BOS"], 0.0) ]
        
        for i, word in enumerate(sent):
            new_states = []
            
            # Capitalization flag
            C = False
            if self._C and word[0].isupper():
                C = True
                
            # Determine possible tags for this word
            if word in self._wd:
                tags = self._wd[word].keys()
            elif self._unk:
                # Use standard NLTK unk tagger if available
                tags = [t for w, t in self._unk.tag([word])]
            else:
                tags = ["UNK"]

            # Expand states
            for history, curr_sent_logprob in current_states:
                t1 = history[-2]
                t2 = history[-1]
                
                for t in tags:
                    tC = (t, C) # Next tag
                    
                    # Probabilities
                    p_uni = self._uni.freq(tC)
                    p_bi = self._bi[t2].freq(tC)
                    p_tri = self._tri[(t1, t2)].freq(tC)
                    
                    curr_uni_count = self._uni[tC]
                    if curr_uni_count == 0:
                        p_wd = 0
                    else:
                        if word in self._wd and t in self._wd[word]:
                             p_wd = self._wd[word][t] / curr_uni_count
                        else:
                             p_wd = 1e-5
                    
                    # Linear Interpolation
                    p = self._l1 * p_uni + self._l2 * p_bi + self._l3 * p_tri
                    
                    if p == 0: p = 1e-10 
                    if p_wd == 0: p_wd = 1e-10
                        
                    p2 = math.log(p, 2) + math.log(p_wd, 2)
                    total_score = curr_sent_logprob + p2
                    new_states.append( (history + [tC], total_score) )
            
            # PRUNING: Keep top N
            if len(new_states) > self.N:
                new_states.sort(key=lambda x: x[1], reverse=True)
                current_states = new_states[:self.N]
            else:
                current_states = new_states
                
            # Fail-safe if empty
            if not current_states:
                 current_states = [ (history + [('NNC', C)], -9999.0) for history, _ in current_states ]
        
        if not current_states:
            return []
            
        best_path = max(current_states, key=lambda x: x[1])[0]
        
        final_tags = []
        for i, word in enumerate(sent):
            tag_node = best_path[i+2]
            if isinstance(tag_node, tuple):
                final_tags.append( (word, tag_node[0]) )
            else:
                final_tags.append( (word, tag_node) )
                
        return final_tags

## 3. Model Loading Function

In [3]:
def load_tnt_model(lex_file, ngram_file):
    pos_tagger = BeamTnT(N=100) 
    pos_tagger._C = False 
    C_params = False 
    
    print("Loading Lexicon...")
    try:
        with open(lex_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('%%'): continue
                parts = line.split()
                if len(parts) < 2: continue
                    
                word = parts[0]
                tag_counts = parts[2:]
                for i in range(0, len(tag_counts), 2):
                    if i + 1 >= len(tag_counts): break
                    tag = tag_counts[i]
                    is_sinhala_tag = any('\u0d80' <= c <= '\u0dff' for c in tag)
                    if is_sinhala_tag: continue
                         
                    try:
                        count = int(tag_counts[i+1])
                        pos_tagger._wd[word][tag] += count
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error loading lexicon: {e}")
        return None

    print("Loading N-Grams...")
    try:
        with open(ngram_file, 'r', encoding='utf-8') as f:
            last_uni = None
            last_bi_t1 = None
            last_bi_t2 = None
            
            def clean_tag(t):
                if t == "__$": return "BOS"
                return t

            for line in f:
                if line.startswith('%%'): continue
                stripped = line.lstrip('\t')
                level = len(line) - len(stripped)
                parts = stripped.split()
                if len(parts) < 2: continue
                
                tag = clean_tag(parts[0])
                try: count = int(parts[1])
                except ValueError: continue
                
                key_node = (tag, C_params)
                if level == 0:
                    pos_tagger._uni[key_node] = count
                    last_uni = tag 
                elif level == 1:
                    if last_uni:
                         pos_tagger._bi[(last_uni, C_params)][key_node] = count
                         last_bi_t1, last_bi_t2 = last_uni, tag
                elif level == 2:
                    if last_bi_t1 and last_bi_t2:
                        pos_tagger._tri[((last_bi_t1, C_params), (last_bi_t2, C_params))][key_node] = count
    except Exception as e:
        print(f"Error loading n-grams: {e}")
        return None

    # Validation fix
    print("Validating model consistency...")
    missing_tags = set()
    for word, tag_freqs in pos_tagger._wd.items():
        for tag, count in tag_freqs.items():
            if pos_tagger._uni[(tag, C_params)] == 0:
                missing_tags.add(tag)
    
    for tag in missing_tags:
        calc_count = sum(tag_freqs[tag] for word, tag_freqs in pos_tagger._wd.items() if tag in tag_freqs)
        pos_tagger._uni[(tag, C_params)] = calc_count

    pos_tagger._N = sum(pos_tagger._uni.values())
    pos_tagger._l1, pos_tagger._l2, pos_tagger._l3 = 0.1, 0.3, 0.6
    return pos_tagger

## 4. Run Tagging (Step 1)
Loads JSON tokens and tags them, storing the results in a structured dictionary: `tagged_data[century][sentence_index] = [(word, tag), ...]`

In [4]:
# Configuration
json_path = "../Data/century_tokens.json"
csv_path = "../Data/century_morphology_data_v6.csv"
output_path = "../data/century_morphology_data_tnt_full.csv"
lex_path = "Sinhala-POS-Tagger/sinhala_final.lex"
ngram_path = "Sinhala-POS-Tagger/sinhala_final.123"

# Load Model
tagger = load_tnt_model(lex_path, ngram_path)

# Load JSON Data
with open(json_path, 'r', encoding='utf-8') as f:
    century_tokens = json.load(f)

century_order = ['5', '12', '13', '14', '15', '16', '18', '19', '20']

# Structure: tagged_data['5'][0] = [('word', 'tag'), ...]
tagged_data = defaultdict(dict)
total_tagged_tokens = 0

print("Starting tagging...")

LIMIT_SENTENCES = None 

for century in century_order:
    if century not in century_tokens: continue
    sentences = century_tokens[century]
    print(f"Processing Century {century} ({len(sentences)} sentences)...")
    
    for s_idx, sentence in enumerate(sentences):
        if LIMIT_SENTENCES and s_idx >= LIMIT_SENTENCES:
            break
            
        try:
            tagged = tagger.tag(sentence)
            tagged_data[century][s_idx] = tagged
            total_tagged_tokens += len(tagged)
        except Exception as e:
            print(f"Error tagging sentence {century}:{s_idx}: {e}")

print(f"Tagging complete. Total valid tokens tagged: {total_tagged_tokens}")

Loading Lexicon...
Loading N-Grams...
Validating model consistency...
Starting tagging...
Processing Century 5 (31 sentences)...
Processing Century 12 (62 sentences)...
Processing Century 13 (624 sentences)...
Processing Century 14 (114 sentences)...
Processing Century 15 (581 sentences)...
Processing Century 16 (343 sentences)...
Processing Century 18 (238 sentences)...
Processing Century 19 (872 sentences)...
Processing Century 20 (1529 sentences)...
Tagging complete. Total valid tokens tagged: 64813


## 5. Alignment and Saving (Step 2)
Loads CSV, groups rows by sentence, aligns them with tagged data, and writes output.

In [5]:
print("Reading CSV...")
csv_rows_out = []
matches_found = 0

with open(csv_path, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    if 'pos_tnt' not in fieldnames:
        fieldnames.append('pos_tnt')
        
    # Group CSV rows by (century, sentence_index)
    # This is safer than stream processing if ordering is slightly off, 
    # though we assume general order is maintained.
    # Actually, let's stream-process providing we track the current key.
    
    current_key = None # (century, sentence_index)
    current_rows = []
    
    def process_sentence_batch(rows, c_key):
        global matches_found
        century, s_idx = c_key
        
        # Get tagged tokens for this sentence
        # Data is str, convert indices if needed. JSON keys are str, sentence indices are int in list.
        # In loop above: tagged_data[century][s_idx]
        # CSV s_idx is string. Convert to int.
        try:
            s_idx_int = int(s_idx)
        except:
             s_idx_int = -1
             
        tagged_sent = tagged_data.get(century, {}).get(s_idx_int, [])
        
        # Alignment Logic for this single sentence
        t_idx = 0
        for row in rows:
            csv_word = row.get('word', '')
            tag_to_assign = ''
            
            if t_idx < len(tagged_sent):
                json_word, json_tag = tagged_sent[t_idx]
                
                if csv_word == json_word:
                    tag_to_assign = json_tag
                    matches_found += 1
                    t_idx += 1
                else:
                    # Mismatch. 
                    # Check 1: Is this CSV row garbage? (Implied if we skip it and find match later)
                    # Check 2: Did JSON miss a token?
                    
                    # Peak ahead: Does this CSV word match next JSON token?
                    if t_idx + 1 < len(tagged_sent) and csv_word == tagged_sent[t_idx+1][0]:
                         # JSON had an extra token that CSV missed? No, that would mean we skip JSON token.
                         # If csv_word matches NEXT json token, we skipped one JSON token.
                         tag_to_assign = tagged_sent[t_idx+1][1]
                         matches_found += 1
                         t_idx += 2 # Skip current and consume next
                    else:
                        # Assume CSV row is garbage
                        pass
            
            row['pos_tnt'] = tag_to_assign
            csv_rows_out.append(row)

    for row in reader:
        c = row['century']
        s = row['sentence_index']
        key = (c, s)
        
        if key != current_key:
            if current_key is not None:
                process_sentence_batch(current_rows, current_key)
            current_key = key
            current_rows = []
        
        current_rows.append(row)
        
    # Process last batch
    if current_key and current_rows:
        process_sentence_batch(current_rows, current_key)

print(f"Alignment Complete. Matches: {matches_found}/{total_tagged_tokens}")
if matches_found < total_tagged_tokens:
    print(f"INFO: {total_tagged_tokens - matches_found} tokens were in JSON but not matched in CSV (normal if CSV has fewer tokens).")

print(f"Writing output to {output_path}...")
with open(output_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_rows_out)
    
print("Done.")

Reading CSV...
Alignment Complete. Matches: 64813/64813
Writing output to ../data/century_morphology_data_tnt_full.csv...
Done.


In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("../data/century_morphology_data_tnt_full.csv")

In [8]:
df[df["pos_tnt"].isna() == True]["word"].value_counts()

Series([], Name: count, dtype: int64)

In [9]:
df["pos_tnt"].value_counts()

pos_tnt
UNK        23180
NNC        11556
VP          5724
POST        4527
JJ          3065
VNF         2414
NNP         2356
DET         1829
RP          1552
PRP         1454
VFM          909
NUM          762
CC           736
VNN          707
NIP          603
NNJ          497
ABB          482
JCV          467
RB           395
NCV          358
AUX          273
CM           258
RPCV         179
RRPCV        120
QBE           93
???           90
PUNC          64
NVB           46
PCV           43
QUE           34
NNC‍          15
1             12
NDT            4
2              3
NNNP           2
NNP‍           2
POST588        1
NNc            1
Name: count, dtype: int64

In [10]:
list(df["pos_tnt"].value_counts().to_dict().keys())

['UNK',
 'NNC',
 'VP',
 'POST',
 'JJ',
 'VNF',
 'NNP',
 'DET',
 'RP',
 'PRP',
 'VFM',
 'NUM',
 'CC',
 'VNN',
 'NIP',
 'NNJ',
 'ABB',
 'JCV',
 'RB',
 'NCV',
 'AUX',
 'CM',
 'RPCV',
 'RRPCV',
 'QBE',
 '???',
 'PUNC',
 'NVB',
 'PCV',
 'QUE',
 'NNC\u200d',
 '1',
 'NDT',
 '2',
 'NNNP',
 'NNP\u200d',
 'POST588',
 'NNc']

In [11]:
pos_map = {
    'NOUN': [
        'NNC',      # Common Nouns [cite: 195]
        'NNJ',      # Adjectival Noun [cite: 90]
        'NUM',      # Number/Numeral [cite: 18]
        'NVB',      # Nominalized Verb [cite: 208]
        'NNc',      # Variant/Typo for NNC
        'NNC\u200d', # NNC with Zero Width Joiner
        'NNNP'      # Likely variant of NNP/NNC [cite: 203]
    ],
    'PROPER NOUN': [
        'NNP',      # Proper Noun [cite: 203]
        'NNP\u200d'   # NNP with Zero Width Joiner
    ],
    'PRONOUN': [
        'PRP',      # Pronoun [cite: 199]
        'QUE',      # Questioning Pronouns [cite: 209]
        'NDT',      # Deterministic Pronoun [cite: 212]
        'QBE'       # Question Based Pronoun [cite: 216]
    ],
    'Verb': [
        'VFM',      # Verb Finite [cite: 42]
        'VNF',      # Verb Non-Finite [cite: 43]
        'VNN',      # Verbal Noun [cite: 43]
        'VP',       # Verb Participle [cite: 43]
        'AUX',      # Auxiliary Verb [cite: 43]
        'NCV',      # Noun in Compound Verb [cite: 64]
        'JCV',      # Adjective in Compound Verb (similar to ACV) [cite: 67]
        'RPCV',     # Relative Participle in Compound Verb
        'RRPCV',    # Relative Participle in Compound Verb
        'PCV'       # Participle in Compound Verb
    ],
    'ADJECTIVE': [
        'JJ'        # Adjective [cite: 189]
    ],
    'PART': [
        'NIP',      # Nipāta (General Particle) [cite: 191]
        'POST',     # Postposition [cite: 120]
        'DET',      # Determiner [cite: 122]
        'RP',       # Particle [cite: 191]
        'CC',       # Conjunction [cite: 191]
        'CM',       # Case Marker [cite: 191]
        'POST588'   # Variant/Error for POST
    ],
    'FOREIGN WORD': [
        'ABB',      # Abbreviation [cite: 155]
        'RB',       # Adverb (Level 1 category in guide) [cite: 190]
        'PUNC',     # Punctuation [cite: 155]
        '1',        # Numeric/Token marker
        '2'         # Numeric/Token marker
    ],
    'UNKNOWN': [
        'UNK',      # Unknown word
        '???',      # Unidentified symbol/token
    ]
}

In [12]:
df["tnt_upd_pos"] = df["pos_tnt"].map({tag: k for k, v in pos_map.items() for tag in v})

In [13]:
df.head(20)

,century,sentence_index,word,new_lemma,morphology,pos,all_tags,all_lemmas,all_morphologies,pos_tnt,tnt_upd_pos
0,5,0,කාම,කාම,කාම+N+NUTR+PL+NOM,NOUN,NOUN,කාම,කාම+N+NUTR+PL+NOM,JJ,ADJECTIVE
1,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN
2,5,0,රූප,රූප,රූප+N+NUTR+PL+NOM,NOUN,NOUN,රූප,රූප+N+NUTR+PL+NOM,NNC,NOUN
3,5,0,ලොක,ලො,ලො+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,ලො,ලො+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN
4,5,0,අරූපලෝක,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,NOUN,NOUN,අරූපලෝ,අරූපලෝ+GUESS+N+F+SG+ID+ACC,UNK,UNKNOWN
5,5,0,යන්,යන්,+N+F+PL+ACC,NOUN,NOUN,යන්,+N+F+PL+ACC,NNC,NOUN
6,5,0,ලොකත්රයට,ලොකත්රයට,NaN,Proper noun or Invalid,Proper noun or Invalid,ලොකත්රයට,NaN,UNK,UNKNOWN
7,5,0,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ+GUESS+N+RT,NOUN,NOUN,ශ්රෙෂ්ඨ,ශ්රෙෂ්ඨ+GUESS+N+RT,UNK,UNKNOWN
8,5,0,වූ,ව,ව+V+NFIN+PASP,Verb,Verb,ව,ව+V+NFIN+PASP,VP,Verb
9,5,0,මුනීන්ද්ර,මුනීන්ද්ර,NaN,Proper noun or Invalid,Proper noun or Invalid,මුනීන්ද්ර,NaN,UNK,UNKNOWN


In [14]:
df.to_csv("../data/century_morphology_data_tnt_full.csv", index=False, encoding='utf-8-sig')